# (QLora) Fine-tuning Mistral-7b-Instruct to Respond to YouTube Comments

Code authored by: Shaw Talebi <br>
Video link: https://youtu.be/XpoKB3usmKc <br>
Blog link: https://towardsdatascience.com/qlora-how-to-fine-tune-an-llm-on-a-single-gpu-4e44d6b5be32 <br>

GitHub: https://github.com/ShawhinT/YouTube-Blog/tree/main/LLMs/qlora

### imports

In [1]:
!pip install auto-gptq
!pip install optimum
!pip install bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.5/23.5 MB 54.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.1/314.1 kB 40.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 54.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.2/13.2 MB 95.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 32.6 MB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl (731.7 MB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl (410.6 MB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl (121.6 MB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_6

In [2]:
# resolving "No inf checks were recorded for this optimizer." issue
!pip uninstall torch -y
!pip install torch==2.1

Found existing installation: torch 2.3.0+cu121
Uninstalling torch-2.3.0+cu121:
  Successfully uninstalled torch-2.3.0+cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 670.2/670.2 MB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.8/209.8 MB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.2/89.2 MB 9.6 MB/s eta 0:00:00
  Attempting uninstall: triton
    Found existing installation: triton 2.3.0
    Uninstalling triton-2.3.0:
      Successfully uninstalled triton-2.3.0
  Attempting uninstall: nvidia-nccl-cu12
    Found existing installation: nvidia-nccl-cu12 2.20.5
    Uninstalling nvidia-nccl-cu12-2.20.5:
      Successfully uninstalled nvidia-nccl-cu12-2.20.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.3.0+cu121 requires torch==2.3.0, but you have torch 2.1.0 which is incompatible.
torchtext 0.18.0

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from peft import prepare_model_for_kbit_training
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
import transformers

### Load model

In [4]:
model_name = "TheBloke/Mistral-7B-Instruct-v0.2-GPTQ"
model = AutoModelForCausalLM.from_pretrained(model_name,
                                             device_map="auto", # automatically figures out how to best use CPU + GPU for loading model
                                             trust_remote_code=False, # prevents running custom model files on your machine
                                             revision="main") # which version of model to use in repo

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.08k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.16G [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/modeling_utils.py:4481: FutureWarning: `_is_quantized_training_enabled` is going to be deprecated in transformers 4.39.0. Please use `model.hf_quantizer.is_trainable` instead
  warnings.warn(
Some weights of the model checkpoint at TheBloke/Mistral-7B-Instruct-v0.2-GPTQ were not used when initializing MistralForCausalLM: ['model.layers.0.mlp.down_proj.bias', 'model.layers.0.mlp.gate_proj.bias', 'model.layers.0.mlp.up_proj.bias', 'model.layers.0.self_attn.k_proj.bias', 'model.layers.0.self_attn.o_proj.bias', 'model.layers.0.self_attn.q_proj.bias', 'model.layers.0.self_attn.v_proj.bias', 'model.layers.1.mlp.down_proj.bias', 'model.layers.1.mlp.gate_proj.bias', 'model.layers.1.mlp.up_proj.bias', 'model.layers.1.self_attn.k_proj.bias', 'model.layers.1.self_attn.o_proj.bias', 'model.layers.1.self_attn.q_proj.bias', 'model.layers.1.self_attn.v_proj.bias', 'model.layers.10.mlp.down_proj.bias', 'model.layers.10.mlp.gate_proj.bias', 'model.la

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [5]:
# Check device allocation of the model parameters
for name, param in model.named_parameters():
    print(f"Parameter {name} is on device: {param.device}")

# Alternatively, check if the model is using GPU
if next(model.parameters()).is_cuda:
    print("The model is using GPU.")
else:
    print("The model is using CPU.")

Parameter model.embed_tokens.weight is on device: cuda:0
Parameter model.layers.0.input_layernorm.weight is on device: cuda:0
Parameter model.layers.0.post_attention_layernorm.weight is on device: cuda:0
Parameter model.layers.1.input_layernorm.weight is on device: cuda:0
Parameter model.layers.1.post_attention_layernorm.weight is on device: cuda:0
Parameter model.layers.2.input_layernorm.weight is on device: cuda:0
Parameter model.layers.2.post_attention_layernorm.weight is on device: cuda:0
Parameter model.layers.3.input_layernorm.weight is on device: cuda:0
Parameter model.layers.3.post_attention_layernorm.weight is on device: cuda:0
Parameter model.layers.4.input_layernorm.weight is on device: cuda:0
Parameter model.layers.4.post_attention_layernorm.weight is on device: cuda:0
Parameter model.layers.5.input_layernorm.weight is on device: cuda:0
Parameter model.layers.5.post_attention_layernorm.weight is on device: cuda:0
Parameter model.layers.6.input_layernorm.weight is on device:

### Load tokenizer

In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

tokenizer_config.json:   0%|          | 0.00/1.46k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

### Using Base Model

In [7]:
model.eval() # model in evaluation mode (dropout modules are deactivated)

# craft prompt
comment = "How many were scored in over 0 ball 1?"
prompt=f'''[INST] {comment} [/INST]'''

# tokenize input
inputs = tokenizer(prompt, return_tensors="pt")

# generate output
outputs = model.generate(input_ids=inputs["input_ids"].to("cuda"), max_new_tokens=140)

print(tokenizer.batch_decode(outputs)[0])

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


<s> [INST] How many were scored in over 0 ball 1? [/INST] I'm unable to answer that question as it's not clear what you mean by "over 0 ball 1." Could you please provide more context or clarify your question? For example, are you asking about the number of points scored in a game of basketball, and if so, which team or teams were playing, and was there a specific inning or quarter being referred to? Without this information, it's impossible to provide an accurate answer.</s>


#### Prompt Engineering

In [8]:
intstructions_string = f"""

Please respond to the following comment and if you dont know the answer return dont know.
"""

prompt_template = lambda comment: f'''[INST] {intstructions_string} \n{comment} \n[/INST]'''

prompt = prompt_template(comment)
print(prompt)

[INST] 

Please respond to the following comment and if you dont know the answer return dont know.
 
How many were scored in over 0 ball 1? 
[/INST]


In [9]:
# tokenize input
inputs = tokenizer(prompt, return_tensors="pt")

# generate output
outputs = model.generate(input_ids=inputs["input_ids"].to("cuda"), max_new_tokens=140)

print(tokenizer.batch_decode(outputs)[0])

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


<s> [INST] 

Please respond to the following comment and if you dont know the answer return dont know.
 
How many were scored in over 0 ball 1? 
[/INST] I'm unable to provide an answer to that question without more context. The term "over 0 ball 1" is not a standard term in sports or mathematics, and it's not clear what you mean by "scored in over 0 ball 1." Could you please clarify what you mean by "over 0 ball 1" and what you mean by "scored"? This will help me provide a more accurate response. If you don't get a response soon, feel free to ask again for clarification. I'll be here to help.</s>


### Prepare Model for Training

In [ ]:
model.train() # model in training mode (dropout modules are activated)

# enable gradient check pointing
model.gradient_checkpointing_enable()

# enable quantized training
model = prepare_model_for_kbit_training(model)

In [ ]:
# LoRA config
config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# LoRA trainable version of model
model = get_peft_model(model, config)

# trainable parameter count
model.print_trainable_parameters()

trainable params: 2,097,152 || all params: 264,507,392 || trainable%: 0.7929


### Preparing Training Dataset

In [ ]:
# load dataset
data = load_dataset("shawhin/shawgpt-youtube-comments")

Generating train split:   0%|          | 0/50 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/9 [00:00<?, ? examples/s]

In [ ]:
print(data)

DatasetDict({
    train: Dataset({
        features: ['example'],
        num_rows: 50
    })
    test: Dataset({
        features: ['example'],
        num_rows: 9
    })
})


In [ ]:
train_size = len(data['train'])
test_size = len(data['test'])

In [13]:
import json
from datasets import Dataset, DatasetDict

# Load the JSON file
with open('/content/1415751.json', 'r') as file:
    cricket_data = json.load(file)

# Create a list to store the examples with over and ball information
examples = []

# Process the JSON data to extract relevant information
for inning in cricket_data['innings']:
    for over in inning['overs']:
        over_number = over['over']
        for ball_number, delivery in enumerate(over['deliveries'], start=1):
            batter = delivery['batter']
            bowler = delivery['bowler']
            runs = delivery['runs']['total']

            # Format the comment with over number and ball number inside [INST] [INST]
            comment = f"<s>[INST] In over {over_number}, ball {ball_number}[/INST] {batter} faced {bowler} and scored {runs} run(s). </s>"
            examples.append(comment)

# Create Dataset
dataset = Dataset.from_dict({'example': examples})

# Print column names and some examples
print(dataset.column_names)  # Output: ['example']
print(dataset[:5])  # Output first 5 examples

# Save the dataset to disk if needed
dataset.save_to_disk('cricket_dataset1')

# To view more details about the dataset
print(dataset)  # This will give a summary of the dataset

# To view a specific example
print(dataset[0])  # This will print the first example in the dataset

# You can also convert the dataset to a Pandas DataFrame for better visualization
import pandas as pd

df = pd.DataFrame(dataset)
print(df.head())  # View the first few rows of the DataFrame


['example']
{'example': ['<s>[INST] In over 0, ball 1[/INST] RG Sharma faced MA Starc and scored 0 run(s). </s>', '<s>[INST] In over 0, ball 2[/INST] RG Sharma faced MA Starc and scored 0 run(s). </s>', '<s>[INST] In over 0, ball 3[/INST] RG Sharma faced MA Starc and scored 4 run(s). </s>', '<s>[INST] In over 0, ball 4[/INST] RG Sharma faced MA Starc and scored 1 run(s). </s>', '<s>[INST] In over 0, ball 5[/INST] V Kohli faced MA Starc and scored 0 run(s). </s>']}


Saving the dataset (0/1 shards):   0%|          | 0/245 [00:00<?, ? examples/s]

Dataset({
    features: ['example'],
    num_rows: 245
})
{'example': '<s>[INST] In over 0, ball 1[/INST] RG Sharma faced MA Starc and scored 0 run(s). </s>'}
                                             example
0  <s>[INST] In over 0, ball 1[/INST] RG Sharma f...
1  <s>[INST] In over 0, ball 2[/INST] RG Sharma f...
2  <s>[INST] In over 0, ball 3[/INST] RG Sharma f...
3  <s>[INST] In over 0, ball 4[/INST] RG Sharma f...
4  <s>[INST] In over 0, ball 5[/INST] V Kohli fac...


In [12]:
dataset

Dataset({
    features: ['example'],
    num_rows: 245
})

In [ ]:
# create tokenize function
def tokenize_function(examples):
    # extract text
    text = examples["example"]

    #tokenize and truncate text
    tokenizer.truncation_side = "left"
    tokenized_inputs = tokenizer(
        text,
        return_tensors="np",
        truncation=True,
        max_length=512
    )

    return tokenized_inputs

# tokenize training and validation datasets
tokenized_data = data.map(tokenize_function, batched=True)

Map:   0%|          | 0/208 [00:00<?, ? examples/s]

Map:   0%|          | 0/37 [00:00<?, ? examples/s]

In [ ]:
# setting pad token
tokenizer.pad_token = tokenizer.eos_token
# data collator
data_collator = transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)


### Fine-tuning Model

In [ ]:
# hyperparameters
lr = 2e-4
batch_size = 4
num_epochs = 10

# define training arguments
training_args = transformers.TrainingArguments(
    output_dir= "shawgpt-ft",
    learning_rate=lr,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_epochs,
    weight_decay=0.01,
    logging_strategy="epoch",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    gradient_accumulation_steps=4,
    warmup_steps=2,
    fp16=True,
    optim="paged_adamw_8bit",

)

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
# configure trainer
trainer = transformers.Trainer(
    model=model,
    train_dataset=tokenized_data["train"],
    eval_dataset=tokenized_data["test"],
    args=training_args,
    data_collator=data_collator
)


# train model
model.config.use_cache = False  # silence the warnings. Please re-enable for inference!
trainer.train()

# renable warnings
model.config.use_cache = True

/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:429: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,3.511800,2.324508
2,1.476300,1.654485
3,0.998700,1.393047
4,0.769000,1.417913
5,0.690500,1.452731
6,0.665100,1.676619
7,0.646500,1.634664
8,0.625700,1.758652
9,0.609600,1.745911
10,0.598900,1.778736


/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:429: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=T

### Push model to hub

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

# # option 2: key login
# from huggingface_hub import login
# write_key = 'hf_' # paste token here
# login(write_key)

In [ ]:
hf_name = 'Yashwanth-07' # your hf username or org name
model_id = hf_name + "/" + "cricket"

In [ ]:
model.push_to_hub(model_id)
trainer.push_to_hub(model_id)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


adapter_model.safetensors:   0%|          | 0.00/8.40M [00:00<?, ?B/s]

events.out.tfevents.1720199577.69c64bdb57dd.310.0:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/8.40M [00:00<?, ?B/s]

Upload 3 LFS files:   0%|          | 0/3 [00:00<?, ?it/s]

training_args.bin:   0%|          | 0.00/5.11k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/Yashwanth-07/shawgpt-ft/commit/d73358f8631d89d9624116958ccb3cb5eed65493', commit_message='Yashwanth-07/cricket', commit_description='', oid='d73358f8631d89d9624116958ccb3cb5eed65493', pr_url=None, pr_revision=None, pr_num=None)

### Load Fine-tuned Model

In [ ]:
# load model from hub
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM

model_name = "TheBloke/Mistral-7B-Instruct-v0.2-GPTQ"
model = AutoModelForCausalLM.from_pretrained(model_name,
                                             device_map="auto",
                                             trust_remote_code=False,
                                             revision="main")

config = PeftConfig.from_pretrained("Yashwanth-07/cricket")
model = PeftModel.from_pretrained(model, "Yashwanth-07/cricket")

# load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

/usr/local/lib/python3.10/dist-packages/transformers/modeling_utils.py:4481: FutureWarning: `_is_quantized_training_enabled` is going to be deprecated in transformers 4.39.0. Please use `model.hf_quantizer.is_trainable` instead
  warnings.warn(
Some weights of the model checkpoint at TheBloke/Mistral-7B-Instruct-v0.2-GPTQ were not used when initializing MistralForCausalLM: ['model.layers.0.mlp.down_proj.bias', 'model.layers.0.mlp.gate_proj.bias', 'model.layers.0.mlp.up_proj.bias', 'model.layers.0.self_attn.k_proj.bias', 'model.layers.0.self_attn.o_proj.bias', 'model.layers.0.self_attn.q_proj.bias', 'model.layers.0.self_attn.v_proj.bias', 'model.layers.1.mlp.down_proj.bias', 'model.layers.1.mlp.gate_proj.bias', 'model.layers.1.mlp.up_proj.bias', 'model.layers.1.self_attn.k_proj.bias', 'model.layers.1.self_attn.o_proj.bias', 'model.layers.1.self_attn.q_proj.bias', 'model.layers.1.self_attn.v_proj.bias', 'model.layers.10.mlp.down_proj.bias', 'model.layers.10.mlp.gate_proj.bias', 'model.la

adapter_config.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/8.40M [00:00<?, ?B/s]

### Use Fine-tuned Model

In [ ]:
intstructions_string = f"""
Please respond to the following comment and if you dont know the answer return dont know.
"""
comment = "How many runs did Rohit Sharma Score in 2007 T20 WC final?"
prompt_template = lambda comment: f'''[INST] {intstructions_string} \n{comment} \n[/INST]'''

prompt = prompt_template(comment)
print(prompt)

[INST] 
Please respond to the following comment and if you dont know the answer return dont know.
 
How many runs did Rohit Sharma Score in 2007 T20 WC final? 
[/INST]


In [ ]:
model.eval()

inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(input_ids=inputs["input_ids"].to("cuda"), max_new_tokens=280)

print(tokenizer.batch_decode(outputs)[0])

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


<s> [INST] 
Please respond to the following comment and if you dont know the answer return dont know.
 
How many runs did Rohit Sharma Score in 2007 T20 WC final? 
[/INST] I don't have the exact answer to that question. Rohit Sharma did not play in the 2007 T20 WC final. The man of the match was Shoaib Malik from Pakistan, who scored 72 runs.</s>


In [ ]:
comment = "What is fat-tailedness?"
prompt = prompt_template(comment)

model.eval()
inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(input_ids=inputs["input_ids"].to("cuda"), max_new_tokens=280)
print(tokenizer.batch_decode(outputs)[0])

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


<s> [INST] 
Please respond to the following comment and if you dont know the answer return dont know.
 
What is fat-tailedness? 
[/INST] Fat-tailedness is a property of certain stochastic processes, specifically those with heavy tails in their probability distributions. In finance, fat-tailedness refers to the presence of extreme price movements that are more frequent and have larger magnitudes than what is predicted by a normal distribution. These extreme events can lead to significant financial risk. In statistics, fat-tailedness is often modeled using distributions other than the normal distribution, such as the Student's t-distribution or the Generalized Pareto Distribution.</s>
